# Keeper244: explore one session

Everything the video needs is made by `python -m analysis.figures <session folders>` (from `python/`).
This notebook is for looking around in one session: the tables, a block, a custom plot.
Session files: `docs/SESSION_LOGS.md`. To try it before the lab: `python tools/make_sim_session.py --short`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # python/ (run Jupyter from python/analysis or python/)
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from analysis.session import load
from analysis import figures, style
style.apply()

sessions = sorted(pathlib.Path('../../data/sessions').glob('*')) or sorted(pathlib.Path('../data/sessions').glob('*'))
print('\n'.join(map(str, sessions[-10:])))

In [ ]:
s = load(sessions[-1])      # pick another with load('…/data/sessions/<folder>')
print(s.participant, s.fs, 'Hz,', round(len(s.x) / s.fs / 60, 1), 'min', '(SIMULATED)' if s.simulated else '')
print('blocks:', s.blocks())
s.steps.head(12)

## The protocol blocks and the game
`s.steps` has every calibration and protocol step; `s.shots` one row per shot with the live filter profile.

In [ ]:
print(s.shots.groupby('profile')[['bridge_saved', 'unity_saved']].mean().round(3) if not s.shots.empty else 'no shots')
figures.game_table(s) if not s.shots.empty else None

## A block, raw and filtered

In [ ]:
t0, t1 = s.block('biceps') or (s.raw_t[0], s.raw_t[0] + 20)
x = s.raw_between(t0, t0 + 20)
y = figures.filtered(s, x)
tt = np.arange(len(x)) / s.fs
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(tt, x[:, 0] / 1e-6 - np.nanmean(x[:, 0]) / 1e-6, color=style.INK_3, lw=0.5, label='raw')
ax.plot(tt, y[:, 0] / 1e-6, color=style.MUSCLE['biceps'], lw=0.5, label='filtered')
ax.set_xlabel('s'); ax.set_ylabel('µV'); ax.legend();

## The same samples through every filter profile (offline)

In [ ]:
from emg_bridge.offline import process
i0, i1 = s.raw_index(t0, t0 + 30)
for prof in s.cal.names if s.cal else []:
    r = process(s.x[i0:i1], s.fs, s.meta['config'], prof, s.cal, n0=int(s.n[i0]))
    print(f"{prof:9s} slot changes: {np.count_nonzero(np.diff(r['slot'])):4d}   time off centre: {np.mean(r['slot'] != 0) * 100:5.1f} %")

## All the figures for this session

In [ ]:
summary = figures.analyse(s.dir, pathlib.Path('../../figures'))
summary['skipped']